In [1]:

import pandas as pd
from scipy.io import savemat
import os
import matlab.engine

def generate_time_sequence(length):
    return [i / 96 for i in range(length)]


def create_experiment_kla_sequence(
    days,
    nominal_kla,
    kla_fraction,
    DR_len,
    tank=5,
    start_day_offset=0,
    repeat_every_days=5,
    nominal_kla_tank3=None,
):
    steps_per_day = 96
    n_initial_nominal = 32
    n_DR = DR_len * 4  # Convert hours to 15-minute steps
    n_final_nominal = steps_per_day - n_initial_nominal - n_DR
    if n_final_nominal < 0:
        raise ValueError(
            "DR_len is too long; reduce the experiment duration or adjust the daily schedule."
        )
    if not 0 <= start_day_offset < repeat_every_days:
        raise ValueError("start_day_offset must be within [0, repeat_every_days).")

    if tank == 5 and nominal_kla_tank3 is not None:
        tank_nominal_kla = nominal_kla_tank3
    else:
        tank_nominal_kla = nominal_kla

    if not 0 < kla_fraction <= 1:
        raise ValueError("kla_fraction must be between 0 and 1 (exclusive).")

    day_experiment = (
        [tank_nominal_kla] * n_initial_nominal
        + [tank_nominal_kla * kla_fraction] * n_DR
        + [tank_nominal_kla] * n_final_nominal
    )
    day_nominal = [tank_nominal_kla] * steps_per_day

    sequence = []
    for day in range(days):
        if day >= start_day_offset and (day - start_day_offset) % repeat_every_days == 0:
            sequence.extend(day_experiment)
        else:
            sequence.extend(day_nominal)
    return sequence

def save_kla_to_mat(kla_sequence, tank, tag):
    time_seq = generate_time_sequence(len(kla_sequence))
    df = pd.DataFrame({'Sequence': time_seq, 'Value': kla_sequence})
    combined = df[['Sequence', 'Value']].values.tolist()

    var_name = f'KLa{tank}_Setpoints_ASM3'
    filename = f'/Users/ikai/github/WWDR/ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/KLa{tank}_Setpoints_{tag}.mat'
    savemat(filename, {var_name: combined})
    print(f"Saved: {filename}")


def create_experimental_kla_files(
    experiment_length,
    days=609,
    start_day_offset=0,
    repeat_every_days=3,
    kla_fraction=0.25,
):
    """Create experimental KLa files for all tanks with given experiment length."""
    nominal_kla_tank3 = 84
    kla_nominal_value = 240

    print(
        f"\nCreating experimental KLa files for experiment_length = {experiment_length} hours "
        f"(start offset = {start_day_offset}, repeat every {repeat_every_days} days)"
    )

    for tank in [3, 4, 5]:
        # Generate experimental sequence
        experiment_seq = create_experiment_kla_sequence(
            days,
            kla_nominal_value,
            kla_fraction,
            experiment_length,
            tank=tank,
            start_day_offset=start_day_offset,
            repeat_every_days=repeat_every_days,
            nominal_kla_tank3=nominal_kla_tank3,
        )

        # Save experimental .mat file
        save_kla_to_mat(experiment_seq, tank=tank, tag="experiment")

    print(f"✓ Completed experimental KLa files for {experiment_length}-hour DR events")

def run_ASM3_data_generation():
    """Run the ASM3 DR data generation MATLAB script"""
    print("\nStarting MATLAB ASM3 data generation...")
    
    try:
        # Start MATLAB engine
        eng = matlab.engine.start_matlab()
        
        # Change to the correct directory (update this path as needed)
        eng.cd('/Users/ikai/github/WWDR/ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3', nargout=0)
        
        # Run the simplified ASM3 DR data generation script
        eng.run('ASM3_DR_datagen.m', nargout=0)
        
        # Quit MATLAB
        eng.quit()
        
        print("✓ MATLAB data generation completed successfully")
        
    except Exception as e:
        print(f"✗ Error during MATLAB execution: {str(e)}")
        raise

def backup_results(experiment_length, start_day_offset, kla_fraction):
    """Backup results for current experiment length"""
    backup_dir = f"/Users/ikai/github/WWDR-Databases/Databases/ExpLength_{experiment_length}h_startday_{start_day_offset}_reductionfactor_{kla_fraction}pct"
    
    try:
        if not os.path.exists(backup_dir):
            os.makedirs(backup_dir)
        
        import shutil
        
        # Copy ASM3_OutputDB directory contents to backup
        source_output_dir = "/Users/ikai/github/WWDR/ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/ASM3_OutputDB"
        if os.path.exists(source_output_dir):
            shutil.copytree(source_output_dir, f"{backup_dir}/ASM3_OutputDB", dirs_exist_ok=True)
            print(f"✓ ASM3_OutputDB backed up to {backup_dir}/ASM3_OutputDB")
        else:
            print(f"⚠ Warning: Output directory {source_output_dir} not found")
        
        # Copy DR_images directory contents to backup
        source_images_dir = "/Users/ikai/github/WWDR/ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/DR_images"
        if os.path.exists(source_images_dir):
            shutil.copytree(source_images_dir, f"{backup_dir}/DR_images", dirs_exist_ok=True)
            print(f"✓ DR_images backed up to {backup_dir}/DR_images")
        else:
            print(f"⚠ Warning: Images directory {source_images_dir} not found")
            
        print(f"✓ Complete results backup completed for {experiment_length}h experiment")
            
    except Exception as e:
        print(f"⚠ Warning: Could not backup results: {str(e)}")


In [ ]:


def main():
    """Main automation routine for multiple DR dataset generation"""

    print("=" * 60)
    print("AUTOMATED DR DATASETS GENERATION")
    print("=" * 60)
    print("Generating datasets for experiment lengths: 0–12 hours")

    start_length = 0
    end_length = 12
    offset_set = [0]
    kla_fractions = [1.0, 0.25, 0.5, 0.75, 0.95]

    lengths = list(range(start_length, end_length + 1))
    total_datasets = len(lengths) * len(offset_set) * len(kla_fractions)
    print(f"Total datasets to generate: {total_datasets}")
    print("=" * 60)

    dataset_counter = 0
    for offset_idx, start_day_offset in enumerate(offset_set, 1):
        print(f"\n--- Offset {offset_idx}/{len(offset_set)}: start day {start_day_offset} ---")

        for fraction in kla_fractions:
            print(f"\n*** KLa fraction {fraction} ***")
            for experiment_length in lengths:
                dataset_counter += 1
                print("\n" + "=" * 50)
                print(
                    f"DATASET {dataset_counter}/{total_datasets}: {experiment_length}-HOUR DR events "
                    f"(start day {start_day_offset}, fraction {fraction})"
                )
                print("=" * 50)

                try:
                    create_experimental_kla_files(
                        experiment_length,
                        start_day_offset=start_day_offset,
                        kla_fraction=fraction,
                    )
                    run_ASM3_data_generation()
                    backup_results(experiment_length, start_day_offset, fraction)
                    print(f"\n✓ Dataset {dataset_counter}/{total_datasets} completed successfully")
                except Exception as e:
                    print(f"\n✗ Error in dataset {dataset_counter}: {str(e)}")
                    print("Stopping automation due to error.")
                    return

    print("\n" + "=" * 60)
    print("AUTOMATION COMPLETED")
    print("=" * 60)
    print(
        f"Generated {total_datasets} datasets across offsets {offset_set}, "
        f"experiment lengths {lengths}, and KLa fractions {kla_fractions}"
    )
    print("Results are backed up in separate directories for each experiment length")
    print("Check individual Results_ExpLength_*h directories for outputs")

if __name__ == "__main__":
    main()




AUTOMATED DR DATASETS GENERATION
Generating datasets for experiment lengths: 0–12 hours
Total datasets to generate: 65

--- Offset 1/1: start day 0 ---

*** KLa fraction 1.0 ***

DATASET 1/65: 0-HOUR DR events (start day 0, fraction 1.0)

Creating experimental KLa files for experiment_length = 0 hours (start offset = 0, repeat every 3 days)
Saved: /Users/ikai/github/WWDR/ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/KLa3_Setpoints_experiment.mat
Saved: /Users/ikai/github/WWDR/ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/KLa4_Setpoints_experiment.mat
Saved: /Users/ikai/github/WWDR/ASM1-2d-3_MATLAB/ASM1 2d 3 in BSM1/ASM3/KLa5_Setpoints_experiment.mat
✓ Completed experimental KLa files for 0-hour DR events

Starting MATLAB ASM3 data generation...

=== ASM3 BENCHMARK MODEL ===
Initializing workspace and model parameters...
Configuration complete:
  - Simulation duration: 273.0 days
  - Calibration period: 245.0 days
  - Segment duration: 14.0 days
  - Total segments: 2

=== PHASE 1b: STEADY STATE INI